# A. Student Performance Predictor (Google Colab)

Notebook ini menjalankan pipeline machine learning end-to-end untuk memprediksi kelas performa akhir siswa pada mata pelajaran Matematika:

* **Low**: nilai akhir G3 <= 9
* **Medium**: 10 <= G3 <= 13
* **High**: G3 >= 14

Sumber dataset: [UCI Machine Learning Repository — Student Performance (ID 320)](https://archive.ics.uci.edu/dataset/320/student+performance), file `student-mat.csv` (395 siswa, 33 atribut, pemisah `;`).

Model yang dilatih: **Logistic Regression** dan **Random Forest**. Model terbaik dipilih berdasarkan **F1-macro** karena distribusi kelas tidak seimbang.

# B. Koneksi ke Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## B1. Konfigurasi path dataset

Ubah `DATA_PATH` di bawah ini jika lokasi file `student-mat.csv` di Google Drive Anda berbeda. Semua output (model & hasil prediksi) disimpan ke `OUTPUT_DIR`.

In [ ]:
import os

# ==== UBAH BAGIAN INI SESUAI LOKASI DATASET ANDA ====
DATA_PATH = '/content/drive/MyDrive/student-performance/student-mat.csv'
OUTPUT_DIR = '/content/drive/MyDrive/student-performance'
# =====================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di: {DATA_PATH}\n"
        "Solusi:\n"
        "1. Pastikan Google Drive sudah ter-mount (jalankan cell di atas).\n"
        "2. Upload file 'student-mat.csv' ke Google Drive.\n"
        "3. Sesuaikan nilai DATA_PATH dengan lokasi file tersebut."
    )
print("Dataset ditemukan:", DATA_PATH)

## B2. Load library dan dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# file student-mat.csv asli dari UCI menggunakan pemisah titik-koma (;)
df = pd.read_csv(DATA_PATH, sep=';')

# --- validasi sederhana: pastikan kolom penting tersedia ---
REQUIRED_COLUMNS = [
    'school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
    'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures',
    'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet',
    'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health',
    'absences', 'G3',
]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Kolom berikut tidak ditemukan di dataset: {missing_cols}\n"
        "Pastikan file yang digunakan adalah 'student-mat.csv' asli dari UCI "
        "(pemisah kolom ';', 33 kolom)."
    )
if not pd.api.types.is_numeric_dtype(df['G3']) or not df['G3'].between(0, 20).all():
    raise ValueError("Kolom target 'G3' harus berupa angka dalam rentang 0-20.")

print(f"Dataset valid: {df.shape[0]} baris x {df.shape[1]} kolom")
df.head()

In [ ]:
df.info()

In [ ]:
# cek missing values
df.isnull().sum()

# C. Preprocessing

## C1. Tahap 1: Tentukan fitur dan label

* Label (target) dibuat dari kolom nilai akhir **G3** yang dikonversi menjadi 3 kelas: Low / Medium / High.
* Kolom **G1 dan G2 tidak dipakai sebagai fitur** karena keduanya adalah nilai periode sebelumnya yang hampir langsung menentukan G3 — menggunakannya adalah *data leakage* (nilai tersebut tidak tersedia saat prediksi nyata dilakukan).
* Fitur yang dipakai: 30 kolom demografi, kebiasaan belajar, dan gaya hidup.

In [ ]:
# konversi nilai akhir G3 (0-20) menjadi kelas performa
def grade_to_class(g3):
    if g3 <= 9:
        return 'Low'
    if g3 <= 13:
        return 'Medium'
    return 'High'

label_df = df['G3'].map(grade_to_class)

# fitur = semua kolom kecuali G1, G2 (leakage) dan G3 (target asli)
fitur_df = df.drop(columns=['G1', 'G2', 'G3'])

print("Jumlah fitur:", fitur_df.shape[1])
label_df.value_counts()

In [ ]:
# visualisasi distribusi kelas target
plt.figure(figsize=(5, 4))
sns.countplot(x=label_df, order=['Low', 'Medium', 'High'])
plt.title('Distribusi Kelas Performa')
plt.xlabel('Kelas')
plt.ylabel('Jumlah Siswa')
plt.show()

## C2. Tahap 2: Bagi dataset menjadi data latih dan data uji

Dataset dibagi 80% data latih dan 20% data uji dengan `stratify` agar proporsi kelas Low/Medium/High tetap seimbang di kedua bagian (penting karena kelasnya tidak seimbang).

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    fitur_df, label_df, test_size=0.2, random_state=RANDOM_STATE, stratify=label_df
)

In [ ]:
print("Banyak data latih (fitur):", len(X_train))
print("Banyak data latih (label):", len(y_train))
print("Banyak data uji (fitur)  :", len(X_test))
print("Banyak data uji (label)  :", len(y_test))

## C3. Tahap 3: Bangun preprocessing pipeline

* Kolom **numerik**: imputasi median (+ StandardScaler untuk model yang sensitif terhadap skala, mis. Logistic Regression).
* Kolom **kategorikal**: imputasi modus + one-hot encoding (`drop='if_binary'` agar kolom biner hanya menjadi satu kolom 0/1).

Preprocessing dipasang di dalam `Pipeline` sklearn sehingga hanya di-fit pada data latih — mencegah data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_COLS = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
    'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences',
]
BINARY_CATEGORICAL_COLS = [
    'school', 'sex', 'address', 'famsize', 'Pstatus',
    'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
    'higher', 'internet', 'romantic',
]
MULTI_CATEGORICAL_COLS = ['Mjob', 'Fjob', 'reason', 'guardian']
CATEGORICAL_COLS = BINARY_CATEGORICAL_COLS + MULTI_CATEGORICAL_COLS

def build_preprocessor(scale_numeric):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), NUMERIC_COLS),
        ('cat', categorical_pipe, CATEGORICAL_COLS),
    ])

## C4. Tahap 4: Siapkan model dan lakukan training

Dua model dibandingkan:

1. **Logistic Regression** — model linear sederhana sebagai baseline (dengan scaling fitur).
2. **Random Forest** — model ensemble berbasis pohon yang mampu menangkap relasi non-linear.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

pipelines = {
    'logistic_regression': Pipeline([
        ('preprocess', build_preprocessor(scale_numeric=True)),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    'random_forest': Pipeline([
        ('preprocess', build_preprocessor(scale_numeric=False)),
        ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
}

results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = {'pipeline': pipeline, 'y_pred': y_pred}
    print(f"Training selesai: {name}")

# D. Evaluasi Model

## D1. Perbandingan akurasi dan F1-macro

F1-macro menghitung F1 setiap kelas lalu dirata-rata tanpa bobot, sehingga model tidak bisa "menang" hanya dengan bagus di kelas mayoritas.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

comparison = pd.DataFrame({
    name: {
        'accuracy': accuracy_score(y_test, res['y_pred']),
        'f1_macro': f1_score(y_test, res['y_pred'], average='macro'),
    }
    for name, res in results.items()
}).T
comparison

## D2. Pilih model terbaik (berdasarkan F1-macro)

In [ ]:
best_name = comparison['f1_macro'].idxmax()
best_pipeline = results[best_name]['pipeline']
y_pred_best = results[best_name]['y_pred']

print(f"Model terbaik: {best_name} "
      f"(F1-macro {comparison.loc[best_name, 'f1_macro']:.3f}, "
      f"accuracy {comparison.loc[best_name, 'accuracy']:.3f})")

## D3. Confusion matrix model terbaik

In [ ]:
from sklearn.metrics import confusion_matrix

CLASSES = ['Low', 'Medium', 'High']
cm = confusion_matrix(y_test, y_pred_best, labels=CLASSES)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## D4. Classification report model terbaik

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_best, labels=CLASSES, zero_division=0))

# E. Simpan Model dan Hasil Prediksi

Model terbaik disimpan sebagai satu `Pipeline` utuh (preprocessing + model) dengan joblib, sehingga saat inference cukup diberi data mentah dengan kolom yang sama. Hasil prediksi data uji juga disimpan sebagai CSV.

In [ ]:
import joblib
import numpy as np

os.makedirs(OUTPUT_DIR, exist_ok=True)

# simpan model terbaik
model_path = os.path.join(OUTPUT_DIR, 'model.joblib')
joblib.dump(best_pipeline, model_path)
print("Model tersimpan di:", model_path)

# simpan fitur + prediksi + label asli dari data uji
hasil_lengkap = pd.concat([
    X_test.reset_index(drop=True),
    pd.DataFrame({'prediksi': y_pred_best, 'label_asli': y_test.reset_index(drop=True)}),
], axis=1)
hasil_path = os.path.join(OUTPUT_DIR, 'hasil_prediksi.csv')
hasil_lengkap.to_csv(hasil_path, index=False)
print("Hasil prediksi tersimpan di:", hasil_path)

# sanity check: load ulang model dan pastikan prediksinya konsisten
reloaded = joblib.load(model_path)
assert np.array_equal(reloaded.predict(X_test.head(5)), best_pipeline.predict(X_test.head(5)))
print("Reload check: OK")

# F. Contoh Prediksi Satu Siswa

Contoh penggunaan model yang sudah disimpan untuk memprediksi satu data siswa (diambil dari data uji).

In [ ]:
sample = X_test.head(1)
pred_class = reloaded.predict(sample)[0]
pred_proba = reloaded.predict_proba(sample)[0]

print("Prediksi performa:", pred_class)
for cls, prob in sorted(zip(reloaded.classes_, pred_proba), key=lambda x: x[1], reverse=True):
    print(f"  {cls}: {prob:.1%}")

# G. Kesimpulan

1. Dataset `student-mat.csv` (395 siswa) berhasil diproses menjadi masalah klasifikasi 3 kelas (Low/Medium/High) berdasarkan nilai akhir G3, tanpa menggunakan G1/G2 untuk menghindari data leakage.
2. Dua model dibandingkan: Logistic Regression dan Random Forest. Model terbaik dipilih berdasarkan **F1-macro**, bukan hanya akurasi, karena distribusi kelas tidak seimbang.
3. Nilai performa yang diperoleh tergolong moderat — hal ini wajar karena fitur prediktor terkuat (nilai periode sebelumnya) sengaja tidak digunakan; model hanya mengandalkan faktor demografi, kebiasaan belajar, dan gaya hidup.
4. Model terbaik disimpan sebagai pipeline utuh (`model.joblib`) dan dapat langsung dipakai untuk prediksi data baru, misalnya pada aplikasi Streamlit.